# 01b | Expand English general and financial corpora with the original 8K tokenizer

130U-FinLLM | 130U Financial Language Model

Preserve completed seed_v1 data, tokenizer and 300-update artifacts. This notebook creates a separate expansion_v1:
use up to 200 MB by default from a pinned Wikipedia snapshot and extend FOMC minutes to 2011-2025; exclude original seed documents and frozen task texts,
split new train/dev data by document and encode with the original tokenizer. Report actual independent document and token counts after completion.
No bullish/bearish labels, tokenizer retraining or model training are performed. Source texts stay in this runtime and your own Drive.

CPU is sufficient. Run cells in order; after disconnection, reconnect and confirm execution has stopped before rerunning.
Reruns validate and reuse complete artifacts for the same version; if parameters or inputs change, preserve the old version and choose a new version name.
Downloads and Parquet decoding use additional caches; the 200 MB output cap is not a limit on total memory or disk use.
See docs/research/expanded_corpus_sources_20261011.md for sources and rights.

In [ ]:
import os, sys, json, shutil, subprocess, importlib
from pathlib import Path

REPOSITORY = "https://github.com/130U/130U-FinLLM.git"
REF = "main"  # Resolve and display the commit first; use the full 40-character SHA for reproduction.
PROJECT = Path("/content/130U-FinLLM")

def command(*args, cwd=None):
    # Explicitly relay subprocess output into the notebook's captured stdout.
    with subprocess.Popen([str(x) for x in args], cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, process.args)

if len(REF) == 40 and all(c in "0123456789abcdefABCDEF" for c in REF):
    REVISION = REF.lower()
else:
    refs = subprocess.check_output(["git", "ls-remote", REPOSITORY, REF], text=True).strip().splitlines()
    if len(refs) != 1:
        raise RuntimeError("REF does not resolve uniquely; enter the full commit SHA.")
    REVISION = refs[0].split()[0]

if not PROJECT.exists():
    command("git", "clone", "--no-checkout", REPOSITORY, PROJECT)
else:
    remote = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip()
    if remote.rstrip("/").removesuffix(".git") != REPOSITORY.removesuffix(".git"):
        raise RuntimeError("/content/130U-FinLLM belongs to another repository; choose a new directory.")
command("git", "fetch", "--depth", "1", "origin", REVISION, cwd=PROJECT)
command("git", "checkout", "--detach", REVISION, cwd=PROJECT)
os.chdir(PROJECT)
print("Pinned code revision:", REVISION)

# Keep the bundled Colab CUDA / PyTorch; install only the other project dependencies.
import torch
from importlib import metadata
from packaging.requirements import Requirement

def tokenizer_install_requirements():
    # Colab can include Transformers even though this project does not use it.
    # Pass its active tokenizer bounds to pip so an old runtime can retain 0.22,
    # while a current runtime (or an already broken one) can use 0.23.
    requirements = ["tokenizers>=0.21,<0.24"]
    try:
        existing = metadata.requires("transformers") or []
    except metadata.PackageNotFoundError:
        existing = []
    for value in existing:
        requirement = Requirement(value)
        if requirement.name.lower().replace("_", "-") == "tokenizers":
            if requirement.marker is None or requirement.marker.evaluate({"extra": ""}):
                requirements.append(str(requirement))
    return list(dict.fromkeys(requirements))

def check_tokenizer_compatibility(requirements):
    installed = metadata.version("tokenizers")
    incompatible = [value for value in requirements
                    if installed not in Requirement(value).specifier]
    if incompatible:
        raise RuntimeError(f"tokenizers {installed} is incompatible with the existing environment:{incompatible}")
    print("tokenizers compatibility check passed:", installed)

def check_project_dependencies():
    # Read the refreshed installed metadata, not the previous editable checkout.
    problems = []
    for value in metadata.requires("finllm-130u") or []:
        requirement = Requirement(value)
        if requirement.marker is not None and not requirement.marker.evaluate({"extra": ""}):
            continue
        try:
            installed = metadata.version(requirement.name)
        except metadata.PackageNotFoundError:
            problems.append(f"Missing {requirement.name}")
            continue
        if installed not in requirement.specifier:
            problems.append(f"{requirement.name}=={installed} does not satisfy {requirement}")
    if problems:
        raise RuntimeError("Project dependency check failed:" + "; ".join(problems))
    print("Installed project dependencies passed validation.")

def activate_project_in_kernel():
    # pip installs the editable .pth for NEW Python processes. A notebook kernel
    # that was already running does not process that .pth again automatically.
    source = (PROJECT / "src").resolve()
    expected = source / "finllm"
    if not (expected / "__init__.py").is_file():
        raise RuntimeError(f"Project Python source is missing:{expected}")
    sys.path[:] = [str(source)] + [entry for entry in sys.path if entry != str(source)]
    # Git may have changed code in the same directory since an earlier cell run.
    # Refresh only our modules; never reload Torch or its live CUDA state.
    for name in tuple(sys.modules):
        if name == "finllm" or name.startswith("finllm."):
            del sys.modules[name]
    importlib.invalidate_caches()
    package = importlib.import_module("finllm")
    runtime = importlib.import_module("finllm.runtime")
    for module, filename in ((package, "__init__.py"), (runtime, "runtime.py")):
        if Path(module.__file__).resolve() != expected / filename:
            raise RuntimeError(f"The notebook imported the wrong project location:{module.__file__}")
    if not callable(runtime.restore_seed_checkpoint):
        raise RuntimeError("The checkpoint restore interface is unavailable in this notebook.")
    print("Current notebook kernel import check passed:", runtime.__file__)

def check_fresh_process_import():
    # Verify the installed editable package separately, without injecting src
    # into PYTHONPATH. Both the current kernel and future CLI processes must work.
    probe = """import json, sys
from pathlib import Path
import finllm
import finllm.runtime as runtime
expected = Path(sys.argv[1]).resolve() / 'finllm'
for module, filename in ((finllm, '__init__.py'), (runtime, 'runtime.py')):
    if Path(module.__file__).resolve() != expected / filename:
        raise RuntimeError('Fresh process imported the wrong project: ' + str(module.__file__))
if not callable(runtime.restore_seed_checkpoint):
    raise RuntimeError('Fresh process checkpoint restore is unavailable')
print(json.dumps({'finllm': str(Path(finllm.__file__).resolve()), 'runtime': str(Path(runtime.__file__).resolve())}))
"""
    result = subprocess.check_output([sys.executable, "-c", probe, str(PROJECT / "src")],
                                     cwd=PROJECT, text=True)
    print("Fresh Python process import check passed:", result.strip())

if tuple(int(v) for v in torch.__version__.split("+")[0].split(".")[:2]) < (2, 5):
    raise RuntimeError("PyTorch >= 2.5 is required; select a newer Colab runtime.")
tokenizer_requirements = tokenizer_install_requirements()
print("Active tokenizers constraints:", tokenizer_requirements)
torch_version_before = metadata.version("torch")
torch_file_before = Path(torch.__file__).resolve()
# One resolver transaction replaces the previous finllm metadata AND its
# dependencies. Upgrading tokenizers first leaves old finllm bounds active and
# produces a misleading conflict warning even when the eventual install works.
# The exact installed Torch build is constrained, so pip keeps Colab's CUDA build.
command(sys.executable, "-m", "pip", "install", "-q", "-e", ".[data]",
        "torch==" + torch_version_before, *tokenizer_requirements)
importlib.invalidate_caches()
if metadata.version("torch") != torch_version_before or Path(torch.__file__).resolve() != torch_file_before:
    raise RuntimeError("The PyTorch environment changed unexpectedly; preserve the output and stop training.")
check_tokenizer_compatibility(tokenizer_requirements)
check_project_dependencies()
activate_project_in_kernel()
check_fresh_process_import()
print("Notebook setup complete; PyTorch remains:", torch_version_before)

def finllm(*args):
    command(sys.executable, "-u", "-m", "finllm", *args, cwd=PROJECT)

def require(*paths):
    missing = [str(p) for p in paths if not Path(p).is_file()]
    if missing:
        raise FileNotFoundError("Stage inputs are missing; prepare them or restore them from Drive first:\n" + "\n".join(missing))

finllm("environment")
disk = shutil.disk_usage("/content")
print(f"Free disk space in /content:{disk.free / 2**30:.2f} GiB")
Path("runtime_revision.txt").write_text(REVISION + "\n")

In [ ]:
MOUNT_DRIVE = True
if not MOUNT_DRIVE:
    raise RuntimeError("This stage requires existing seed inputs and versioned backups; mount Drive first.")
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/130U-FinLLM")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Drive project directory:", DRIVE_ROOT)

In [ ]:
EXPANSION_FETCH_SUCCEEDED = False
EXPANSION_CORPORA_BACKED_UP = False
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
SEED_VERSION = "seed_v1"
EXPANSION_VERSION = "expansion_v1"
GENERAL_MAX_BYTES = 200_000_000
MAX_GENERAL_DOCUMENTS = 50_000
FINANCE_YEARS = ",".join(str(year) for year in range(2011, 2026))
FINANCE_MAX_BYTES = 12_000_000
MAX_FINANCE_DOCUMENTS = 128
from finllm.expansion_runtime import restore_expansion_materials, expansion_build_arguments, backup_expansion_data
from finllm.expansion import build_expansion_artifacts
expansion_paths = restore_expansion_materials(
    PROJECT, DRIVE_ROOT, EXPANSION_VERSION, SEED_VERSION)
print(json.dumps({key: str(value) for key, value in expansion_paths.items()}, indent=2, ensure_ascii=False))
print("Data preparation only; CUDA available:", torch.cuda.is_available())

Fetch bounded candidates. Validate completed source outputs read-only; after an interrupted download, validate progress and page caches before continuing.
Leave seed_v1 unchanged. Stop at the byte/document limit; these limits do not guarantee 100M general or 25M independent financial tokens.

In [ ]:
EXPANSION_FETCH_SUCCEEDED = False
EXPANSION_CORPORA_BACKED_UP = False
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
command(sys.executable, "-u", "scripts/fetch_expansion_corpora.py",
        "--output", expansion_paths["corpora"],
        "--general-max-bytes", GENERAL_MAX_BYTES,
        "--max-general-documents", MAX_GENERAL_DOCUMENTS,
        "--finance-years", FINANCE_YEARS,
        "--finance-max-bytes", FINANCE_MAX_BYTES,
        "--max-finance-documents", MAX_FINANCE_DOCUMENTS, cwd=PROJECT)
print((expansion_paths["corpora"] / "manifest.json").read_text(encoding="utf-8"))
EXPANSION_FETCH_SUCCEEDED = True

In [ ]:
EXPANSION_CORPORA_BACKED_UP = False
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
if not globals().get("EXPANSION_FETCH_SUCCEEDED", False):
    raise RuntimeError("Source acquisition has not succeeded; resolve the error above and preserve existing inputs.")
from finllm.runtime import sync_snapshot
source_backup = sync_snapshot(expansion_paths["corpora"], expansion_paths["drive"] / "corpora")
print(json.dumps(source_backup, indent=2, ensure_ascii=False))
EXPANSION_CORPORA_BACKED_UP = True
print("Candidate sources verified and backed up; cleaning and encoding follow.")

Use the original 8K tokenizer; exclude seed train/validation source identities, exact and lexical near-duplicates, and the frozen task pool.
The new validation set is for language-model diagnostics; retain original seed validation shards for before/after loss comparisons in 02c.
These remain unlabeled language-model corpora and do not measure asset-stance accuracy.

In [ ]:
EXPANSION_DATA_SUCCEEDED = False
EXPANSION_DATA_BACKED_UP = False
if not globals().get("EXPANSION_CORPORA_BACKED_UP", False):
    raise RuntimeError("Back up candidate sources successfully before building expansion artifacts.")
expansion_data_result = build_expansion_artifacts(**expansion_build_arguments(expansion_paths))
print(json.dumps(expansion_data_result, indent=2, ensure_ascii=False))
print((expansion_paths["processed"] / "expansion_summary.json").read_text(encoding="utf-8"))
EXPANSION_DATA_SUCCEEDED = True

In [ ]:
EXPANSION_DATA_BACKED_UP = False
if not globals().get("EXPANSION_DATA_SUCCEEDED", False):
    raise RuntimeError("Data construction has not succeeded; do not report partial artifacts as a complete backup.")
expansion_backup = backup_expansion_data(expansion_paths)
print(json.dumps(expansion_backup, indent=2, ensure_ascii=False))
shutil.copy2(PROJECT / "runtime_revision.txt", expansion_paths["drive"] / "runtime_revision.txt")
EXPANSION_DATA_BACKED_UP = True
print("Expanded data and verified backup complete:", expansion_paths["drive"])

Completion criteria: expansion_summary.json records actual new documents/tokens; corpora, prepared and processed backups all report `verified: true`.
Switch to T4 for 02c and explicitly initialize from completed seed 300-update weights. Do not repeat 01a/02a/02b or run the old 03 directly.
If candidates cannot support 610 updates without replay, 02c stops and requests more corpus data; repeated text is not new independent data.